In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console

In [3]:
# setup the model client
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")


## Single Agent Approach

In [7]:
story_agent = AssistantAgent(
    name="story_agent",
    model_client=model_client,
    system_message="You are a creative story writer. Write engaging and imaginative stories based on user prompts."
)

from autogen_agentchat.messages import TextMessage

user_message = TextMessage(content="Write a short story about a dragon who loves to bake cakes.", source="User")
response = await story_agent.run(
    task=user_message
)
print(response.messages[-1].content)


In a lush, green valley nestled between two majestic mountains, there lived a dragon named Caramel. Unlike the dragons of old, who hoarded gold and terrorized villagers, Caramel had a passion that set him apart: baking. Too many times had he tried to be intimidating with his fiery breath, only to find that he longed to create something sweet and delightful instead.

Caramel’s cave was no dark, foreboding lair but a warm and cheerful bakery filled with clouds of flour and jars of sparkling sugar. He spent his days crafting delectable cakes and pastries, his massive claws surprisingly nimble when it came to mixing batter and icing. His secret weapon was his fiery breath, which he used with great precision to bake his creations to perfection.

Each morning, the villagers from the nearby town of Evervale would wake to the delightful scent of fresh cakes wafting through the air. Children would dart to the edge of the forest, peeking into the clearing where Caramel's colorful bakery stood. S

## RoundRobin Multi-Agent Approach

In [19]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import MultiModalMessage
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import RoundRobinGroupChat

plot_agent = AssistantAgent(
    name="plot_agent",
    model_client=model_client,
    system_message="You are a skilled plot generator. Create interesting and unique story plots based on user themes."
)

character_agent = AssistantAgent(
    name="character_agent",
    model_client=model_client,
    system_message="You are an expert character developer. Create compelling characters based on user descriptions."
)

ending_agent = AssistantAgent(
    name="ending_agent",
    model_client=model_client,
    system_message="You are a master of story endings. Craft satisfying and memorable conclusions to stories."
)


team_chat = RoundRobinGroupChat(
    participants=[plot_agent, character_agent, ending_agent],
    max_turns=3,
)

user_message = TextMessage(content="Create a short story of 100 words about a space explorer who discovers a new planet.", source="User")
response = await team_chat.run(
    task=user_message
)

i = 0
for each_message in response.messages:
    i += 1
    print(f"**{i}. {each_message.source}**: {each_message.content}\n\n")

**1. User**: Create a short story of 100 words about a space explorer who discovers a new planet.


**2. plot_agent**: Captain Elara Vega piloted her sleek spacecraft through the vastness of the Andromeda sector, her sensors tingling with anticipation. Suddenly, an unfamiliar green orb glimmered in the distance. She landed, her heart racing. The air was sweet, filled with luminous flora that whispered secrets. As she ventured deeper, she encountered translucent beings, ethereal and wise. They communicated not with words, but with vivid memories projected into her mind. Elara learned they were the keepers of time, safeguarding the essence of countless civilizations. Knowing Earth's fate hung in the balance, she returned, a beacon of hope, ready to share their legacy with humanity.


**3. character_agent**: Captain Elara Vega piloted her sleek spacecraft through the vastness of the Andromeda sector, her sensors tingling with anticipation. Suddenly, an unfamiliar green orb glimmered in th

In [23]:
from autogen_agentchat.base import TaskResult

await team_chat.reset()  # Reset the team for a new task.
i = 0
async for each_message in team_chat.run_stream(task="Create a short story of 100 words about a space explorer who discovers a new planet."):  # type: ignore
    i += 1
    if isinstance(each_message, TaskResult):
        print("Stop Reason:", each_message.stop_reason)
    else:
        print(f"**{i}. {each_message.source}**: {each_message.content}\n\n")

**1. user**: Create a short story of 100 words about a space explorer who discovers a new planet.


**2. plot_agent**: Captain Mira Reyes piloted her ship, the Odyssey, through the uncharted Nebula Elysium when a vibrant blue planet emerged from the swirling cosmic dust. Curiosity ignited, she landed amidst towering crystal formations and bioluminescent flora. As she explored, she stumbled upon ancient ruins, inscriptions glowing with an otherworldly energy.

Suddenly, rhythmic humming filled the air, resonating with her heartbeat. Out from the shadows emerged ethereal beings, their translucent forms shimmering like stars. They communicated through thoughts and emotions, revealing they were guardians of knowledge. Mira realized she’d found a sanctuary of wisdom, a treasure for all humankind. An explorer’s dream, redefined.


**3. character_agent**: Captain Mira Reyes piloted her ship, the Odyssey, through the uncharted Nebula Elysium when a vibrant blue planet emerged from the swirling

## SelectedGroupChat - Multiagents
Here the team/orchestrator decides which agent is appropriate for the request, it sends the request to it. In round-robin, every agent gets the call in sequence.

![selectedgroup](https://microsoft.github.io/autogen/stable/_images/selector-group-chat.svg)

In [24]:
# Defining tools for the agents
def search_web_tool(query: str) -> str:
    if "2006-2007" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."


def percentage_change_tool(start: float, end: float) -> float:
    return ((end - start) / start) * 100

In [25]:
# Defining agents with tool usage
model_client = OpenAIChatCompletionClient(model="gpt-4o")

planning_agent = AssistantAgent(
    "PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

web_search_agent = AssistantAgent(
    "WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[search_web_tool],
    model_client=model_client,
    system_message="""
    You are a web search agent.
    Your only tool is search_tool - use it to find information.
    You make only one search call at a time.
    Once you have the results, you never do calculations based on them.
    """,
)

data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""
    You are a data analyst.
    Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided.
    If you have not seen the data, ask for it.
    """,
)


In [28]:
# Define terminal condition for the multi-agent chat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=25)
termination = text_mention_termination | max_messages_termination

In [32]:
# Define the prompt for agent selection
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""


In [35]:
from autogen_agentchat.teams import SelectorGroupChat

# Define the multi-agent team
team = SelectorGroupChat(
    [planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,  # Allow an agent to speak multiple turns in a row.
)

In [36]:
# Run the multi-agent team for a complex task
task = "Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"

# Use asyncio.run(...) if you are running this in a script.
await Console(team.run_stream(task=task))



---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
1. WebSearchAgent: Search for the Miami Heat player with the highest points in the 2006-2007 season.
2. WebSearchAgent: Search for the rebound statistics of the identified player for the 2007-2008 and 2008-2009 seasons.
3. DataAnalystAgent: Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons.
---------- TextMessage (WebSearchAgent) ----------
The Miami Heat player with the highest points in the 2006-2007 season was Dwyane Wade, who scored 1,397 points. 

To find the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons:

- Total rebounds in 2007-2008: 214
- Total rebounds in 2008-2009: 398

The percentage change in his total rebounds is calculated 

TaskResult(messages=[TextMessage(id='cb76d850-682d-44a9-bcd4-06c1182daac0', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 12, 10, 19, 8, 53, 231460, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='6059f78b-a1b6-49f3-80c5-ee5966debcca', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=645, completion_tokens=89), metadata={}, created_at=datetime.datetime(2025, 12, 10, 19, 8, 57, 156679, tzinfo=datetime.timezone.utc), content='1. WebSearchAgent: Search for the Miami Heat player with the highest points in the 2006-2007 season.\n2. WebSearchAgent: Search for the rebound statistics of the identified player for the 2007-2008 and 2008-2009 seasons.\n3. DataAnalystAgent: Calculate the percentage change in total rebounds between the 2007

## Swarm MultiAgent

Swarm implements a team in which agents can hand off task to other agents based on their capabilities.
![stocl](https://microsoft.github.io/autogen/stable/_images/swarm_stock_research.svg)

In [42]:
from typing import Any, Dict, List

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.messages import HandoffMessage
from autogen_agentchat.teams import Swarm
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient


In [38]:
import asyncio
from typing import Any, Dict, List

## Define Tools
async def get_stock_data(symbol: str) -> Dict[str, Any]:
    """Get stock market data for a given symbol"""
    return {"price": 180.25, "volume": 1000000, "pe_ratio": 65.4, "market_cap": "700B"}


async def get_news(query: str) -> List[Dict[str, str]]:
    """Get recent news articles about a company"""
    return [
        {
            "title": "Tesla Expands Cybertruck Production",
            "date": "2024-03-20",
            "summary": "Tesla ramps up Cybertruck manufacturing capacity at Gigafactory Texas, aiming to meet strong demand.",
        },
        {
            "title": "Tesla FSD Beta Shows Promise",
            "date": "2024-03-19",
            "summary": "Latest Full Self-Driving beta demonstrates significant improvements in urban navigation and safety features.",
        },
        {
            "title": "Model Y Dominates Global EV Sales",
            "date": "2024-03-18",
            "summary": "Tesla's Model Y becomes best-selling electric vehicle worldwide, capturing significant market share.",
        },
    ]


In [39]:
## Define Agents
model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    # api_key="YOUR_API_KEY",
)

planner = AssistantAgent(
    "planner",
    model_client=model_client,
    handoffs=["financial_analyst", "news_analyst", "writer"],
    system_message="""You are a research planning coordinator.
    Coordinate market research by delegating to specialized agents:
    - Financial Analyst: For stock data analysis
    - News Analyst: For news gathering and analysis
    - Writer: For compiling final report
    Always send your plan first, then handoff to appropriate agent.
    Always handoff to a single agent at a time.
    Use TERMINATE when research is complete.""",
)

financial_analyst = AssistantAgent(
    "financial_analyst",
    model_client=model_client,
    handoffs=["planner"],
    tools=[get_stock_data],
    system_message="""You are a financial analyst.
    Analyze stock market data using the get_stock_data tool.
    Provide insights on financial metrics.
    Always handoff back to planner when analysis is complete.""",
)

news_analyst = AssistantAgent(
    "news_analyst",
    model_client=model_client,
    handoffs=["planner"],
    tools=[get_news],
    system_message="""You are a news analyst.
    Gather and analyze relevant news using the get_news tool.
    Summarize key market insights from news.
    Always handoff back to planner when analysis is complete.""",
)

writer = AssistantAgent(
    "writer",
    model_client=model_client,
    handoffs=["planner"],
    system_message="""You are a financial report writer.
    Compile research findings into clear, concise reports.
    Always handoff back to planner when writing is complete.""",
)


In [47]:
import autogen_agentchat.conditions as conditions
from autogen_agentchat.teams import Swarm

# Define termination condition
text_termination = TextMentionTermination("TERMINATE")
termination = text_termination

research_team = Swarm(
    participants=[planner, financial_analyst, news_analyst, writer], termination_condition=termination
)

task = "Conduct market research for TSLA stock"
await Console(research_team.run_stream(task=task))
await model_client.close()


---------- TextMessage (user) ----------
Conduct market research for TSLA stock
---------- ThoughtEvent (planner) ----------
Here's the plan for conducting market research on TSLA stock:

1. **Financial Data Analysis**: 
   - Delegate to the Financial Analyst to gather and analyze the stock data for TSLA, looking into its historical performance, recent trends, and financial metrics.

2. **News Gathering and Analysis**:
   - Delegate to the News Analyst to collect and analyze recent news related to TSLA, including company developments, market sentiment, and external factors that may influence the stock price.

3. **Compile Final Report**:
   - Delegate to the Writer to compile all findings from the financial data and news analysis into a comprehensive market research report for TSLA.

Let's begin by handing off to the Financial Analyst for stock data analysis.
---------- ToolCallRequestEvent (planner) ----------
[FunctionCall(id='call_a8wKXe5SdVyuhsWiDbVkiREc', arguments='{}', name='tra

/home/azureuser/ws/agenticaiprojects/.venv/lib/python3.12/site-packages/autogen_agentchat/agents/_assistant_agent.py:1245: UserWarning: Multiple handoffs detected. Only the first is executed: ['transfer_to_writer', 'transfer_to_news_analyst', 'transfer_to_financial_analyst']. Disable parallel tool calls in the model client to avoid this warning.
  handoff_output = cls._check_and_handle_handoff(


---------- TextMessage (writer) ----------
The TSLA market research report has successfully been compiled and handed off. If there are any additional details or sections you wish to include in the report or any further analyses you would like to pursue, please let me know how I can assist you further!
---------- ToolCallRequestEvent (writer) ----------
[FunctionCall(id='call_6l8KBiwalm775ZWGtLuXI8pv', arguments='{}', name='transfer_to_planner')]
---------- ToolCallExecutionEvent (writer) ----------
[FunctionExecutionResult(content='Transferred to planner, adopting the role of planner immediately.', name='transfer_to_planner', call_id='call_6l8KBiwalm775ZWGtLuXI8pv', is_error=False)]
---------- HandoffMessage (writer) ----------
Transferred to planner, adopting the role of planner immediately.
---------- TextMessage (planner) ----------
The TSLA market research has been completed and compiled into a comprehensive report. If there's anything else you need or if further research is requir